# Step 1: Setup prerequisites

In [ ]:
import os
import sys
import pymongo
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

In [ ]:
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-agent-memory")
# Check the connection to the server
mongodb_client.admin.command("ping")

In [ ]:
# Set the URL for our AI model proxy
PROXY_ENDPOINT = os.environ.get("PROXY_ENDPOINT")

In [ ]:
# Set the passkey provided by your workshop instructor
PASSKEY = "replace-with-passkey"

In [ ]:
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env(["voyageai"], PASSKEY)

# Step 2: Setup memory collections

In [ ]:
import json
from utils import create_index, create_search_index, check_index_ready

### **Do not change the values assigned to the variables below**

In [ ]:
# Database name
DB_NAME = "mongodb_genai_devday_memory"
# Name of the chat history collection
CHATS_COLLECTION_NAME = "chats"
# Name of the semantic memory collection
SEMANTIC_COLLECTION_NAME = "semantic"
# Name of the procedural memory collection
PROCEDURAL_COLLECTION_NAME = "procedural"
# Name of the vector search index
VS_INDEX_NAME = "vector_index"

📚 **Access a database:** https://www.mongodb.com/docs/languages/python/pymongo-driver/current/databases-collections/#access-a-database

📚 **Access a collection:** https://www.mongodb.com/docs/languages/python/pymongo-driver/current/databases-collections/#access-a-collection

In [ ]:
# Access the `DB_NAME` database.
db = mongodb_client[DB_NAME]
# Access the `CHATS_COLLECTION_NAME` collection.
chats_collection = db[CHATS_COLLECTION_NAME]
# Access the `SEMANTIC_COLLECTION_NAME` collection.
semantic_collection = db[SEMANTIC_COLLECTION_NAME]
# Access the `PROCEDURAL_COLLECTION_NAME` collection.
procedural_collection = db[PROCEDURAL_COLLECTION_NAME]

In [ ]:
# Seed the procedural memories collection with memories
with open(f"../data/memories.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {PROCEDURAL_COLLECTION_NAME} collection.")
procedural_collection.delete_many({})
procedural_collection.insert_many(data)
print(
    f"{procedural_collection.count_documents({})} documents ingested into the {PROCEDURAL_COLLECTION_NAME} collection."
)

# Step 3: Create indexes on the memory collections

In [ ]:
# Use the `create_index` function from the `utils` module to:
# 1. create an index on the "session_id" field
# 2. create a time-to-live (TTL) index to automatically delete documents after 1 year (31536000 seconds)
create_index(chats_collection, ["session_id"], "session_index")
create_index(chats_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

In [ ]:
# Use the `create_index` function from the `utils` module to create a time-to-live (TTL) index on the `semantic_collection` collection
create_index(semantic_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

In [ ]:
# Create the vector index definition for the procedural memories collection, specifying:
# path: Path to the embeddings field
# numDimensions: Number of embedding dimensions- depends on the embedding model used
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [ ]:
# Use the `create_search_index` function from the `utils` module to create a vector search index with the above definition on the `procedural_collection` collection
create_search_index(procedural_collection, VS_INDEX_NAME, model)

In [ ]:
# Use the `check_index_ready` function from the `utils` module to verify that the index was created and is in READY status before proceeding
check_index_ready(procedural_collection, VS_INDEX_NAME)

# Step 4: Define short-term memory management functions

These functions are not exposed as agent tools because chat history needs to be persisted at fixed points in the agent loop (after user input, LLM responses, and tool results) — not based on LLM judgment. We call them directly in code rather than letting the LLM decide when to invoke them.

In [ ]:
from datetime import datetime

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one

In [ ]:
# Define a function to store chat messages to MongoDB
def store_chat_message(session_id: str, role: str, content: str) -> None:
    """
    Store a chat message in a MongoDB collection.

    Args:
        session_id (str): Session ID of the message.
        role (str): Role for the message. One of `user` or `assistant`.
        content (str): Content of the message.
    """
    # Create the message object to persist in MongoDB
    msg_obj = {
        "session_id": session_id,
        "role": role,
        "content": content,
        "timestamp": datetime.now(),
    }
    # Insert the `msg_obj` into the `chats_collection` collection using the `insert_one()` method
    chats_collection.insert_one(msg_obj)
    

📚 **Specifying query filters and projections in find()**: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/project/#exclude-the-_id-field

📚 **Return documens in a specific order**: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/specify-documents-to-return/#sort

In [ ]:
# Define a function to retrieve chat history from MongoDB
def get_chat_history(session_id: str) -> list:
    """
    Retrieve chat history for a session.

    Args:
        session_id (str): Session ID to retrieve chat history for.

    Returns:
        List: List of chat messages.
    """
    # Retrieve session history from the `chats_collection` collection
    # Use the `find()` method to retrieve documents 
    # Filter for documents where the `session_id` field is equal to the provided `session_id`
    # Project only the `role` and `content` fields, exclude the default `_id` field
    # Sort the results by increasing order of `timestamp`  
    cursor =  chats_collection.find({"session_id": session_id}, {"_id": 0, "role": 1, "content": 1}).sort("timestamp", pymongo.ASCENDING)
    docs = list(cursor)
    print("Retrieving chat history...")
    print(f"Found {len(docs)} messages")
    return docs

# Step 5: Define long-term memory management tools

In [ ]:
import requests
import voyageai

In [ ]:
# Initialize the Voyage AI client
vo = voyageai.Client()

📚 https://www.mongodb.com/docs/voyageai/models/text-embeddings/?client=python#example

In [ ]:
# Define a function to generate embeddings using the Atlas Embedding and Reranking API
def get_embedding(text: str, input_type: str) -> list[float]:
    """
    Get embeddings for an input text

    Args:
        text (str): Text to embed
        input_type (str): One of "document" or "query"

    Returns:
        list[float]: Embedding of the query string
    """
    # Use the `embed` method of the Voyage API to embed a piece of text with the following arguments:
    # texts: A list of strings to embed, in this case `text` wrapped in a list.
    # model: `voyage-4`
    # input_type: The `input_type` passed to the `get_embedding` function, which is either "document" or "query".
    embds_obj = vo.embed(texts=[text], model="voyage-4", input_type=input_type)
    # Extract embeddings from the embeddings object (`embds_obj`)
    embeddings = embds_obj.embeddings[0]
    return embeddings

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_many

In [ ]:
# Define a tool to save semantic user memories to MongoDB
def save_user_memories(user_id: str, memories: list[str]) -> str:
    """
    Save user facts and preferences

    Args:
        user_id (str): User ID
        memories (list[str]): Facts and preferences to save

    Returns:
        str: Save message
    """
    docs = []
    # Iterate through the list of user memories to save to MongoDB
    for memory in memories:
        # Create a separate document for each memory
        doc = {
            "user_id": user_id,
            "content": memory,
            "timestamp": datetime.now()
        }
        docs.append(doc)
    # Bulk-insert `docs` into the `semantic_collection` collection using the `insert_many()` method
    semantic_collection.insert_many(docs)
    return f"Saved {len(memories)} memories"

📚 **Specifying query filters and projections in find()**: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/project/#exclude-the-_id-field

📚 **Return documens in a specific order**: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/specify-documents-to-return/#sort

In [ ]:
# Define a tool to get semantic memories from MongoDB
def get_user_memories(user_id: str) -> str:
    """
    Retrieve memories for a particular user

    Args:
        user_id (str): User ID

    Returns:
        str: Retrieved memories formatted as a string
    """
    # Query the `semantic_collection` collection for the current user's memories
    # Use the `find()` method to execute the query 
    # Filter for documents where the `user_id` field is equal to the provided `user_id`
    # Project only the `content` and `timestamp` fields, exclude the default `_id` field
    # Sort the results in decreasing order of `timestamp` 
    cursor = semantic_collection.find({"user_id": user_id}, {"_id": 0, "content": 1, "timestamp": 1}).sort("timestamp", pymongo.DESCENDING)
    docs = list(cursor)
    
    if not docs:
        return f"No memories found for user {user_id}"
    
    # Convert memories into the following format:
    # - 2026-03-12 14:54:13.296000 User prefers MongoDB for data storage
    lines = [f"- {d['timestamp']} {d['content']}" for d in docs]
    return f"Memories for {user_id}:\n" + "\n".join(lines)

In [ ]:
# Path to local scratchpad file
NOTES_FILE = "./NOTES.md"

# Define a tool to write notes to a local scratchpad
def write_notes(notes: list[str]) -> str:
    """
    Write notes to a local file

    Args:
        notes (list[str]): Notes to write to file

    Returns:
        str: Note recorded message
    """
    # Open the notes file in append mode and write each note to the file, prefixed with the timestamp
    with open(NOTES_FILE, "a") as f:
        for note in notes:
            f.write(f"- [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {note}\n")
    return f"Recorded {len(notes)} notes"

📚 **Creating a JSON Schema**: https://json-schema.org/understanding-json-schema/reference/object#required

📚 **Insert a document into MongoDB**: https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one

In [ ]:
# Define a tool to generate procedural memories from notes using an LLM, and persist them to MongoDB
def save_procedure() -> str:
    """
    Generate a procedural memory from notes and save to MongoDB
    
    Returns:
        str: Procedure save acknowledgment
    """
    # Read notes from the scratchpad
    try:
        with open(NOTES_FILE, "r") as f:
            notes_content = f.read().strip()
    except FileNotFoundError:
        return "No notes found to create procedure from."
    
    if not notes_content:
        return "Notes file is empty."
    
    # Use an LLM to generate a procedure title and summary from the notes content. 
    prompt = f"""Based on these session notes, extract a reusable, step-by-step implementation guide that a coding agent could 
    follow to reproduce a similar project from scratch.

    Focus on:
    - The specific sequence of implementation steps
    - Which tools, libraries, and configurations were used and why
    - Key decisions and their rationale
    - Pitfalls encountered and how to avoid them

    Write it as instructions ("do this, then do this"), not as a narrative of what happened.

    Also provide a brief, descriptive title for the procedure.

    Notes: {notes_content}
    """
    # Format the message to the LLM in the format [{"role": <role_value>, "content": <content_value>}]
    # The `role` value for user messages must be "user"
    # Use the `prompt` created above to populate the `content` field in the chat message
    messages = [{"role": "user", "content": prompt}]
    # Create a JSON schema to get a JSON object as output from the LLM, containing only `title` and `description` fields.
    json_schema = {        
        # HINT: The output needs to be a JSON "object"        
        "type": "object",
        # The JSON object should contain "title" and "description" fields
        "properties": {
            "title": {"type": "string", "description": "Brief, descriptive title (5-10 words)"},
            "description": {"type": "string", "description": "A summary of what was accomplished, the approach taken, and key insights (max 2 paragraphs)"},
        },
        # Specify that the "title" and "description" fields are both required
        "required": ["title", "description"],
        # There should be no additional fields in the output so we set the `additionalProperties` field to `False`
        "additionalProperties": False
    }
    # Use the AI model proxy to get an LLM response
    response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {
            "messages": messages,
            "output_config": {"format": {"type": "json_schema", "schema": json_schema}}
        }}
    )
    
    # Parse the LLM response
    procedure = json.loads(response["content"][0]["text"])
        
    # Generate embeddings for the `description` field of the `procedure` using the `get_embedding` function defined above.
    # Specify the `input_type` argument as "document".
    embedding = get_embedding(procedure["description"], "document")
    # Create the procedure document to insert into MongoDB
    procedure_doc = {
        "title": procedure["title"],
        "description": procedure["description"],
        "timestamp": datetime.now(),
        "embedding": embedding
    }
    # Insert the `procedure_doc` into MongoDB
    procedural_collection.insert_one(procedure_doc)
    
    # Clear the notes file after saving the procedure
    open(NOTES_FILE, "w").close()
    
    return f"Procedure saved: '{procedure['title']}' - Notes cleared."

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/#ann-examples (Refer to the "ANN Basic Example")

In [ ]:
# Define a tool to retrieve procedural memories from MongoDB using semantic search
def get_procedures(query: str) -> str:
    """
    Retrieve previous procedures, using semantic search

    Args:
        note (str): The 

    Returns:
        str: Relevant procedures formatted as strings
    """
    # Use the `get_embedding` function defined above to generate an embedding for the input `query`.
    # Specify the `input_type` argument as "query".
    query_embedding = get_embedding(query, "query")
    # Define an aggregation pipeline consisting of a $vectorSearch stage, followed by a $project stage
    # Set the number of candidates to 100 and only return the top 5 documents from the vector search
    # In the $project stage, exclude the `_id` field and include the `title` and `description` fields
    # NOTE: Use variables defined previously for the `index`, `queryVector` and `path` fields in the $vectorSearch stage
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 100,
                "limit": 5
            }
        },
        {"$project": {"_id": 0, "title": 1, "description": 1}}
    ]
    # Execute the aggregation `pipeline` on the `procedural_collection` collection.
    cursor = procedural_collection.aggregate(pipeline)
    docs = list(cursor)
    
    if not docs:
        return "No relevant procedures found."

    # Format the retrieved procedures
    output = "Relevant past procedures:\n"
    for i, ep in enumerate(docs, 1):
        output += f"{i}. {ep['title']}\n"
        output += f"{ep['description']}\n\n"
    return output

# Step 6: Define tool schemas

In [ ]:
# Create JSON schemas for the tools, so the LLM knows what functions it can call, what each function does, and what arguments to provide.
memory_tools = [
    {
        "name": "save_user_memories",
        "description": "Save user preferences or facts as memories.",
        # Guarantee schema validation on tool names and inputs by setting `strict` to True.
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of memories to save"
                },
            },
            "required": ["content"],
            "additionalProperties": False,
        }
    },
    {
        "name": "get_user_memories",
        "description": "Retrieve stored memories for a user. Use at conversation start to get the user's profile.",
        "strict": True,
        "input_schema": {                    
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        }
    },
    {
        "name": "write_notes",
        "description": "Write notes about the current task to a local file.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of notes to write to file."
                },
            },
            "required": ["notes"],
            "additionalProperties": False,
        }
    },
    {
        "name": "save_procedure",
        "description": "Create a procedural memory from accumulated notes.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        }
    },
    {
        "name": "get_procedures",
        "description": "Search past procedural memories by semantic similarity.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Description of the problem or topic to search"
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        }
    }
]

# Step 7: Define tool execution function

In [ ]:
# Define a function to execute the tool calls
# NOTE: LLMs don't handle tool execution. They only determine which tool to execute and the arguments. 
def execute_tool(tool_name: dict, tool_args: dict, session: dict) -> str:
    """
    Coordinate tool execution

    Args:
        tool_name (str): Tool name
        tool_args (dict): Arguments for the tool call
        session (dict): Session information

    Returns:
        str: Tool output formatted as string
    """
    if tool_name == "save_user_memories":
        return save_user_memories(session["user_id"], tool_args["content"])
    elif tool_name == "get_user_memories":
        return get_user_memories(session["user_id"])
    elif tool_name == "write_notes":
        return write_notes(tool_args["notes"])
    elif tool_name == "save_procedure":
        return save_procedure()
    elif tool_name == "get_procedures":
        return get_procedures(tool_args["query"])
    else:
        return f"Unknown tool: {tool_name}"

# Step 8: Create a memory-augmented agent

In [ ]:
import uuid

### Step 8a: Create a session management function

In [ ]:
# Create a session object to maintain state for a user session, that tracks the following:
# session_id: Unique session ID 
# user_id: Current user ID
# input_tokens: Number of input tokens used in the session
# output_tokens: Number of output tokens generated during the session
# max_tokens: The LLM's context window limit
def create_session(user_id: str, max_tokens: int=1000000) -> dict:
    """
    Create a session object to track session metadata

    Args:
        user_id (str): Unique user identifier
        max_tokens (int): The LLM's context window limit

    Returns:
        dict: Session object
    """
    return {
        "session_id": str(uuid.uuid4())[:8],
        "user_id": user_id,
        "input_tokens": 0,
        "output_tokens": 0,
        "max_tokens": max_tokens
    }

### Step 8b: Define a dynamic system prompt for the agent

In [ ]:
# Calculate context window usage during the session-- we will use this as one of the triggers for memory creation
def get_context_usage(session: dict) -> float:
    """
    Calculate context window usage as a percentage

    Args:
        session (dict): Session object

    Returns:
        float: % of the context window used
    """
    total_tokens = session["input_tokens"] + session["output_tokens"]
    return (total_tokens / session["max_tokens"]) * 100

In [ ]:
# Create a system prompt providing guidance on how and when to use the memory tools
def get_system_prompt(session: dict) -> str:
    """
    Build system prompt with context usage status

    Args:
        session (dict): Session object

    Returns:
        str: Agent's system prompt
    """
    usage_pct = get_context_usage(session)
    prompt = f"""You are a helpful coding assistant with memory capabilities.

    ## CURRENT SESSION STATUS
    - Context window usage: {usage_pct:.1f}%

    ## GENERAL INSTRUCTIONS
    - Ask follow-up questions if the user request is unclear or too broad.
    - Keep responses concise and focused. Avoid lengthy code examples unless specifically requested.

    ## MEMORY PROTOCOL
    **At conversation start:**
    - Use `get_user_memories` to retrieve user preferences and facts.
    - Prioritize RECENT information if memories conflict.

    **When to search for past procedures (`get_procedures`):**
    - ANY technical or coding question
    - Procedures are YOUR internal knowledge from past sessions. Use them SILENTLY to inform your approach.

    **During the conversation:**
    - Use `write_notes` to record:
        - Implementation steps taken and their order
        - Tools, libraries, and configurations chosen (and WHY)
        - Errors encountered and how they were resolved
        - User feedback that changed the approach
    - Keep EACH NOTE to 2-3 SENTENCES.
    - Write notes after each meaningful implementation step, not just at the end.

    **When you learn something important about the user:**
    - Use `save_user_memories` to save facts and preferences such as preferred language, frameworks, coding style, and experience level.
    - Keep EACH MEMORY to 1 SENTENCE.

    **When to save a procedure (`save_procedure`):**
    - A multi-step implementation task is COMPLETE — not just a single question answered or concept explained.
    - The solution involved specific tools, configurations, or a sequence of steps that would be useful to reproduce.
    - Do NOT save procedures for simple factual Q&A, single function explanations, or debugging one-liners.
    - Write the procedure as INSTRUCTIONS a future agent could follow ("Set up X, then configure Y"), not as a summary of what happened ("We set up X, then configured Y").
    - Include pitfalls encountered and how to avoid them.
    - DEFINITELY save a procedure when the context window usage exceeds 70%.

    ASSUME INTERRUPTION: Your context window might be reset at any moment, so you risk losing any progress that is not recorded in your notes.
    """
    
    return prompt

### Step 8c: Define the agent loop

In [ ]:
# Define the agent execution loop
def chat(user_input: str, session: dict) -> None:
    """
    The agent execution loop

    Args:
        user_input (str): The user's question
        session (dict): Session object
    """
    session_id = session["session_id"]
    print("===== Human message =====")
    print(user_input)
    # Use the `store_chat_message` function to add the `user_input` to the chat history for this session
    # The `role` value for user messages is "user"
    store_chat_message(session_id, "user", user_input)
    # Run the agent loop until the LLM decides it has the final answer
    while True:
        # Use the `get_system_prompt` function to build a dynamic system prompt with the current context usage status
        system_prompt = get_system_prompt(session)

        # Use the `get_chat_history` function to retrieve chat messages for the current session from MongoDB
        messages = get_chat_history(session_id)
        
        # Send the system prompt, chat history, and the memory tools as input the the LLM
        response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {
            "system": system_prompt,
            "messages": messages,
            "tools": memory_tools
        }})
        if response.status_code == 200:
            result = response.json()
        else:
            # If error in LLM response, print the error message
            print(response.json()["error"])
        
        # Update the session's token usage
        session["input_tokens"] += result["input_tokens"]
        session["output_tokens"] += result["output_tokens"]
        
        content = result["content"]
        print("===== AI message =====")
        text_parts = [block["text"] for block in content if block.get("type") == "text"]
        # Print the text blocks in the LLM's response
        if text_parts:
            print("\n".join(text_parts))
        # Use the `store_chat_message` function to store the LLM response (`content`) in the `chats` collection
        # The `role` value for LLM responses is "assistant"
        store_chat_message(session_id, "assistant", content)
        
        # Final answer reached- break out of the loop
        if result["stop_reason"] == "end_turn":
            break
  
        # Handle tool calls- there can be multiple tool calls in the LLM's response
        elif result["stop_reason"] == "tool_use":
            tool_outputs = []
            for block in content:
                if block.get("type") == "tool_use":
                    # Get the tool name from the LLM response
                    tool_name = block["name"]
                    # Get the tool arguments from the LLM response
                    tool_args = block["input"]
                    print("===== Tool Call =====")
                    print(f"Calling {tool_name} with args: {tool_args}")
                    
                    # Use the `execute_tool` function defined in Step 7 to execute the right tool
                    # Use the`tool_name` and `tool_args` identified by the LLM, and the `session` object as parameters
                    tool_output = execute_tool(tool_name, tool_args, session)
                    print(f"===== Tool Outcome: {tool_name} =====")
                    print(tool_output)
                    
                    # Collect all tool outputs
                    tool_outputs.append({
                        "type": "tool_result",
                        "tool_use_id": block["id"],
                        "content": tool_output
                    })
            # Use the `store_chat_message` function to store the `tool_outputs` in the `chats` collection
            # The `role` value for tool outputs is "user"
            store_chat_message(session_id, "user", tool_outputs)

# Step 9: Interact with the agent

In [ ]:
# Create a new session
session_1 = create_session(user_id="mdb_user")
print(f"Started session: {session_1['session_id']}")

In [ ]:
# Test 1: Ask a coding question
# The agent should take notes and search for relevant past procedures
response = chat("Help me create an AI-powered shopping assistant.", session_1)

In [ ]:
# Test 2: Tell it some of your personal preferences
# The agent should extract and persist semantic memories 
response = chat("MongoDB is my preferred database and Voyage AI is my preferred embedding provider.", session_1)

In [ ]:
# Test 3: Check cross-session behavior
# The agent should recall long-term memories from previous sessions, chat history should get reset
session_2 = create_session(user_id="mdb_user")
print(f"Started session: {session_2['session_id']}")

response = chat("What do you know about me?", session_2)